# Job Recommendation — AI Career Advisor

Builds a TF-IDF + Cosine Similarity recommender, evaluates Precision@K, and saves to `models/recommender.pkl`.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs('../models', exist_ok=True)
jobs_df = pd.read_csv('../data/jobs.csv')
jobs_df['Skills'] = jobs_df['Skills'].fillna('')
print(f'Loaded {jobs_df.shape[0]} job rows, {jobs_df["Job Title"].nunique()} unique roles')
jobs_df.head()

In [ ]:
# ── Build TF-IDF Matrix ────────────────────────────────────────────────────
rec_tfidf  = TfidfVectorizer(max_features=200, ngram_range=(1, 2))
job_matrix = rec_tfidf.fit_transform(jobs_df['Skills'])
print(f'TF-IDF matrix: {job_matrix.shape}')
print(f'Vocabulary size: {len(rec_tfidf.vocabulary_)}')

In [ ]:
# ── Demo Recommendation ────────────────────────────────────────────────────
def recommend_jobs(skills_str, top_n=5, domain=None):
    query_vec = rec_tfidf.transform([skills_str])
    sims = cosine_similarity(query_vec, job_matrix).flatten()
    if domain:
        mask = (jobs_df['Industry'] == domain).values
        sims_filtered = sims.copy()
        sims_filtered[~mask] = 0
    else:
        sims_filtered = sims
    title_scores = {}
    for i, title in enumerate(jobs_df['Job Title'].values):
        if sims_filtered[i] > 0:
            title_scores.setdefault(title, []).append(sims_filtered[i])
    avg = {t: np.mean(v) for t, v in title_scores.items()}
    ranked = sorted(avg.items(), key=lambda x: x[1], reverse=True)[:top_n]
    max_s = ranked[0][1] if ranked else 1
    return [(t, min(s / max(max_s, 1e-6) * 95 + 5, 99)) for t, s in ranked]

sample = 'Python Machine Learning TensorFlow Deep Learning SQL'
recs = recommend_jobs(sample, top_n=5)
print('Sample recommendations for:', sample)
for t, pct in recs:
    print(f'  {pct:.1f}%  {t}')

In [ ]:
# ── Evaluation: Precision@K ────────────────────────────────────────────────
K_values = [1, 3, 5]
np.random.seed(42)
test_idx = np.random.choice(len(jobs_df), min(100, len(jobs_df)//5), replace=False)

precision_at_k = {k: 0 for k in K_values}
recall_at_k    = {k: 0 for k in K_values}

for idx in test_idx:
    query_vec  = job_matrix[idx]
    sims       = cosine_similarity(query_vec, job_matrix).flatten()
    sims[idx]  = 0   # exclude self
    true_title = jobs_df.iloc[idx]['Job Title']
    same_role_indices = set(jobs_df[jobs_df['Job Title'] == true_title].index.tolist()) - {idx}

    for k in K_values:
        top_k_idx = set(np.argsort(sims)[::-1][:k])
        hits  = len(top_k_idx & same_role_indices)
        precision_at_k[k] += hits / k
        recall_at_k[k]    += hits / max(len(same_role_indices), 1)

n = len(test_idx)
print(f"{'K':<5} {'Precision@K':>14} {'Recall@K':>12}")
print('-' * 35)
for k in K_values:
    p = precision_at_k[k] / n
    r = recall_at_k[k] / n
    print(f"@{k:<4} {p:>14.4f} {r:>12.4f}")

In [ ]:
# ── Visualise Similarity Distribution ────────────────────────────────────────
sample_query = rec_tfidf.transform(['Python Machine Learning SQL TensorFlow PyTorch'])
all_sims = cosine_similarity(sample_query, job_matrix).flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(all_sims, bins=50, color='#56b6c2', edgecolor='white', alpha=0.85)
axes[0].set_title('Cosine Similarity Distribution (all jobs)', fontweight='bold')
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Count')

# Top-10 roles bar chart
title_sims = {}
for i, title in enumerate(jobs_df['Job Title'].values):
    title_sims.setdefault(title, []).append(all_sims[i])
avg_sims = pd.Series({t: np.mean(v) for t, v in title_sims.items()}).sort_values(ascending=False).head(10)
axes[1].barh(avg_sims.index[::-1], avg_sims.values[::-1], color='#4f86c6')
axes[1].set_title('Top 10 Matched Roles (avg cosine sim)', fontweight='bold')
axes[1].set_xlabel('Avg Cosine Similarity')
plt.tight_layout()
plt.savefig('../images/recommender_analysis.png', dpi=120)
plt.show()

In [ ]:
# ── Save Recommender ───────────────────────────────────────────────────────
prec5 = precision_at_k[5] / n
rec_bundle = {
    'vectorizer':  rec_tfidf,
    'matrix':      job_matrix,
    'titles':      jobs_df['Job Title'].values,
    'jobs_df':     jobs_df,
    'precision5':  prec5,
}
joblib.dump(rec_bundle, '../models/recommender.pkl')
print('✅ Saved → ../models/recommender.pkl')
print(f'   Precision@5 = {prec5:.3f}')

# Smoke test
loaded = joblib.load('../models/recommender.pkl')
q = loaded['vectorizer'].transform(['Python SQL Machine Learning'])
s = cosine_similarity(q, loaded['matrix']).flatten()
top5 = np.argsort(s)[::-1][:5]
print('Smoke test top-5:', [(loaded['titles'][i], round(float(s[i]),4)) for i in top5])